#  Random Forest Regression Model 3.4

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


## Loading data set, applying preprocessing steps

In [2]:
# Load the dataset
df_popularity = pd.read_csv('tracks2026.csv')
# These two decisions are explained in EDA and Data_Preparation
df_popularity = df_popularity.dropna() 
df_popularity = df_popularity.drop(columns = ["track_id"])

In [3]:
target = 'popularity'
X = df_popularity.drop(columns=[target])
y = df_popularity[target]

In [4]:
# Let us perform the preprosessing, same as in data preparation

# Define column groups
numerical_cols = X.select_dtypes(include=[np.number]).columns
categorical_numerical = ['key', 'mode', 'time_signature', 'explicit']  # numerical but categorical
continuous_numerical = [col for col in numerical_cols if col not in categorical_numerical]

no_outlier_cols = ['valence', 'acousticness']
outlier_cols = [col for col in continuous_numerical if col not in no_outlier_cols]
passthrough_cols = [col for col in df_popularity.columns if col not in continuous_numerical]

# Split outlier columns into negative and positive value groups
outlier_cols_neg = [col for col in outlier_cols if df_popularity[col].min() < 0]
outlier_cols_pos = [col for col in outlier_cols if df_popularity[col].min() >= 0]
# Pipelines
preprocess_no_outliers = Pipeline([
    ('log', FunctionTransformer(func=np.log1p, feature_names_out='one-to-one')),
    ('scaler', StandardScaler())
])

preprocess_outliers_neg = Pipeline([
    ('scaler', RobustScaler())  
])
preprocess_outliers_pos = Pipeline([
     ('log', FunctionTransformer(func=np.log1p, feature_names_out='one-to-one')),
     ('scaler', RobustScaler())
])
# Scale key and time_signature columns, as they have different scale than others
key_timesignature_cols = ["key", "time_signature"]
key_timesignature_pipeline = Pipeline(steps = [
    ('scaler', StandardScaler())
])
minmax_pipeline = Pipeline(steps = [
    ('log', FunctionTransformer(func=np.log1p, feature_names_out='one-to-one')),
    ('scaler', MinMaxScaler())
])
# Do OneHotEncoding here to avoid test set containing information about the track-genre categories in training set. 
# If unknown value, ignore
onehotencode_pipeline = Pipeline(steps = [
    ("onehotencode", OneHotEncoder(handle_unknown='ignore'))
])
preprocess_pipeline = ColumnTransformer(
    transformers=[
        ('no_outliers', preprocess_no_outliers, no_outlier_cols),
        ('outliers_neg', preprocess_outliers_neg, outlier_cols_neg),
        ('outliers_pos', preprocess_outliers_pos, outlier_cols_pos),
        ('key_time_signature', key_timesignature_pipeline, key_timesignature_cols),
        ("onehotencode", onehotencode_pipeline, ["track_genre"] )
    ],
    remainder='passthrough'
)
print(no_outlier_cols)
print(outlier_cols_neg)
print(outlier_cols_pos)
print(key_timesignature_cols)



['valence', 'acousticness']
['loudness']
['duration_ms', 'danceability', 'energy', 'speechiness', 'instrumentalness', 'liveness', 'tempo']
['key', 'time_signature']


## Creating Model and Pipeline

In [ ]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE
from sklearn.model_selection import KFold
from sklearn import svm



# Setting a CV score
cv10 = KFold(n_splits=10, shuffle=True, random_state=42)

# Scoring variable
scoring = {
    "neg_MSE": "neg_mean_squared_error",
    "neg_RMSE": "neg_root_mean_squared_error",
    "neg_MAE": "neg_mean_absolute_error",
    "R2": "r2"
}

# Split the data (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Linear Regression Pipeline
pipe = Pipeline(steps = [('preprocess', preprocess_pipeline),
                         ('reduce_dim', PCA(iterated_power=7)),
                        ('regressor', RandomForestRegressor(random_state=42, n_estimators=10))
                         ]
                         )

# Hyperparameters for GridSearchCV
N_FEATURES_OPTIONS = [2,4,7]
MAX_DEPTH_OPTIONS = [2,3,5,7]

# Parameter grid for GridSearchCV
param_grid = {
    'reduce_dim__n_components': [2, 6, 11, 15, 19],
    'regressor__n_estimators': [10, 50, 100],
    'regressor__max_depth': [None, 10, 20, 30],
    'regressor__min_samples_split': [2, 5, 10]
}


## Applying to model
Applying steps to model and reviewing R2 scores

In [ ]:
from sklearn.model_selection import GridSearchCV


rfr_search = GridSearchCV( # TODO Change this to reflect random forest
    pipe, 
    param_grid,
    scoring=scoring,    
    n_jobs=-1,
    cv=cv10,
    refit="R2",
    return_train_score=False)
# Initialize and train the Random Forest Regressor
rfr_search.fit(X_train, y_train)

print(f"Best parameters: {rfr_search.best_params_}")
print(f"Best R2 score: {rfr_search.best_score_:.4f}")

# Evaluating on test set
y_pred = rfr_search.predict(X_test)
r2_realistic = r2_score(y_test, y_pred)
print("R2 Score on Test Set: ", r2_realistic)

Best parameters: {'reduce_dim__n_components': 19, 'regressor__max_depth': 30, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 50}
Best R2 score: 0.2715
R2 Score on Test Set:  0.2946779803325693


## Conclusion
Using Random Forest Regression we were able to achieve a much better R2 score than we were able to with Linear regression, this could be because Random forest has much less bias.

Applying it to the test set also showed a similar R2 score so we are not as worried about overfitting to the data, as that could be a common issue with Random Forest Regression models.